In [1]:
import os
import nltk
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction import DictVectorizer
from sklearn.svm import LinearSVC
from nltk.classify import MaxentClassifier
from sklearn.metrics import classification_report
import pandas as pd

In [2]:
# Load dataset (assuming it's in a folder with multiple text files)
folder_path = 'test_data'
files = os.listdir(folder_path)
txt_files = [file for file in files if file.endswith('.txt')]
print(len(txt_files))

100


In [3]:
if "kon_agriculture_set1.txt" in txt_files:
    txt_files.remove("kon_agriculture_set1.txt")

print(len(txt_files))

99


In [4]:
# Read and preprocess data
sentences = []
for file in txt_files:
    with open(os.path.join(folder_path, file), 'r', encoding='utf-8') as f:
        sentences.extend(f.readlines())

print(len(sentences))

99191


In [5]:
# Convert sentences into word-tag pairs
def extract_features(sentence):
    features = []
    labels = []
    words = sentence.strip().split()
    for i, token in enumerate(words):
        if "\\" not in token:
            continue  # Skip malformed tokens
        
        parts = token.rsplit("\\", 1)
        if len(parts) != 2:
            continue  # Ensure correct splitting
        
        word, tag = parts
        features.append({
            'word': word.lower(),  # Convert to lowercase to reduce feature space
            'is_first': i == 0,
            'is_last': i == len(words) - 1,
            'is_capitalized': word[0].isupper() if word else False,
            'prefix-2': word[:2] if len(word) > 1 else '',
            'suffix-2': word[-2:] if len(word) > 1 else '',
        })
        labels.append(tag)
    return features, labels

In [6]:
# Prepare dataset
X, y = [], []
for sentence in sentences:
    features, labels = extract_features(sentence)
    X.extend(features)
    y.extend(labels)

In [7]:
## Convert features to numerical format
vectorizer = DictVectorizer(sparse=True)  # Use sparse matrix to save memory
X = vectorizer.fit_transform(X)

In [8]:
## Split data into train and test sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.1, random_state=42)

In [9]:
## Train SVM model with reduced max_iter for efficiency
svm_model = LinearSVC(max_iter=1000)
svm_model.fit(X_train, y_train)


/home/vidyaapati/miniconda3/envs/py38/lib/python3.8/site-packages/sklearn/svm/_classes.py:32: FutureWarning: The default value of `dual` will change from `True` to `'auto'` in 1.5. Set the value of `dual` explicitly to suppress the warning.
  warnings.warn(


LinearSVC()

In [10]:
from sklearn.metrics import classification_report

# Evaluate on training data
y_train_pred = svm_model.predict(X_train)
print("Training Set Evaluation:")
print(classification_report(y_train, y_train_pred))

Training Set Evaluation:


/home/vidyaapati/miniconda3/envs/py38/lib/python3.8/site-packages/sklearn/metrics/_classification.py:1471: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
/home/vidyaapati/miniconda3/envs/py38/lib/python3.8/site-packages/sklearn/metrics/_classification.py:1471: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))


              precision    recall  f1-score   support

                   0.00      0.00      0.00         1
      CC_CCD       1.00      0.99      0.99     29887
      CC_CCS       0.85      0.91      0.88     19976
   CC_CCS_UT       0.00      0.00      0.00        79
    CC_CC_UT       0.00      0.00      0.00       488
       CC_UT       0.00      0.00      0.00       446
      DM_DMD       0.83      0.87      0.85     38513
      DM_DMI       0.67      0.69      0.68      1945
      DM_DMQ       0.53      0.17      0.26       784
      DM_DMR       0.52      0.56      0.54      5560
      DN_DMD       0.00      0.00      0.00         1
          JJ       0.82      0.82      0.82     65031
       N-NNP       1.00      1.00      1.00         1
          NN       0.00      0.00      0.00         4
       NN_NN       1.00      1.00      1.00         1
      NN_NNP       1.00      1.00      1.00         1
         NST       1.00      0.25      0.40         4
        N_NN       0.95    

/home/vidyaapati/miniconda3/envs/py38/lib/python3.8/site-packages/sklearn/metrics/_classification.py:1471: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))


In [11]:
# Predict on validation set
y_pred = svm_model.predict(X_test)
print(classification_report(y_test, y_pred))

/home/vidyaapati/miniconda3/envs/py38/lib/python3.8/site-packages/sklearn/metrics/_classification.py:1471: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
/home/vidyaapati/miniconda3/envs/py38/lib/python3.8/site-packages/sklearn/metrics/_classification.py:1471: UndefinedMetricWarning: Recall and F-score are ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
/home/vidyaapati/miniconda3/envs/py38/lib/python3.8/site-packages/sklearn/metrics/_classification.py:1471: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(res

              precision    recall  f1-score   support

      CC_CCD       1.00      0.99      0.99      3320
      CC_CCS       0.85      0.92      0.89      2268
   CC_CCS_UT       0.00      0.00      0.00        15
    CC_CC_UT       0.00      0.00      0.00        49
       CC_UT       0.00      0.00      0.00        44
      DM_DMD       0.82      0.87      0.84      4238
      DM_DMI       0.66      0.68      0.67       219
      DM_DMQ       0.42      0.09      0.14        93
      DM_DMR       0.48      0.52      0.50       587
          JJ       0.78      0.76      0.77      7182
      NN_NNP       0.00      0.00      0.00         1
        N_NN       0.89      0.95      0.92     42682
       N_NNP       0.83      0.69      0.75      8761
       N_NNV       0.00      0.00      0.00        17
       N_NST       0.84      0.84      0.84      3263
      PR_PRC       0.84      0.77      0.81        35
      PR_PRF       0.96      0.98      0.97       587
      PR_PRI       0.59    

/home/vidyaapati/miniconda3/envs/py38/lib/python3.8/site-packages/sklearn/metrics/_classification.py:1471: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
/home/vidyaapati/miniconda3/envs/py38/lib/python3.8/site-packages/sklearn/metrics/_classification.py:1471: UndefinedMetricWarning: Recall and F-score are ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))


In [12]:
# Function to predict POS tags for a given sentence SVM
def predict_pos(sentence):
    words = sentence.strip().split()
    features = []
    for i, word in enumerate(words):
        features.append({
            'word': word.lower(),
            'is_first': i == 0,
            'is_last': i == len(words) - 1,
            'is_capitalized': word[0].isupper() if word else False,
            'prefix-2': word[:2] if len(word) > 1 else '',
            'suffix-2': word[-2:] if len(word) > 1 else '',
        })
    
    # Convert features to numerical format
    X_features = vectorizer.transform(features)
    predicted_tags = svm_model.predict(X_features)

    #predicted_tags = [memm_model.classify(f) for f in features]
    
    return list(zip(words, predicted_tags))

# Example usage
example_sentence = "ग्रेटर नोयडा वेस्टाचे त्रास कमी जावपाचें नांवूच घेना ."
print(predict_pos(example_sentence))

[('ग्रेटर', 'N_NN'), ('नोयडा', 'N_NNP'), ('वेस्टाचे', 'N_NN'), ('त्रास', 'N_NN'), ('कमी', 'QT_QTF'), ('जावपाचें', 'V_VM_VNF'), ('नांवूच', 'N_NN'), ('घेना', 'V_VM_VF'), ('.', 'RD_PUNC')]


In [13]:
import dill

In [14]:
# Save the tagger using dill
with open("svm_Model.pkl", "wb") as f:
    dill.dump(svm_model, f)

In [15]:
with open("vectorizer_svm.pkl", "wb") as f:
    dill.dump(vectorizer, f)

In [16]:
# Read and preprocess data
sentences = []
for file in txt_files[99:]:
    with open(os.path.join(folder_path, file), 'r', encoding='utf-8') as f:
        sentences.extend(f.readlines())

In [17]:
testData = []
file_path = "test_data/kon_agriculture_set1.txt"

with open(file_path, 'r', encoding='utf-8') as file:
    content = file.read()

content = content.split("\n")[1:]  # Skip the first line
testDataFile = [line.split('\t')[1] for line in content if line and 'Value' not in line.split('\t')[1]]
testData.append(testDataFile)

testTaggedData = [item for sublist in testData for item in sublist]
print(testTaggedData[:2])

['ग्रेटर\\RD_UNK नोयडा\\RD_UNK वेस्टाचे\\RD_UNK त्रास\\N_NN कमी\\QT_QTF जावपाचें\\V_VM_VNF नांवूच\\N_NN घेना\\V_VM_VF .\\RD_PUNC', 'आपल्यो\\PR_PRL मागण्यो\\N_NN घेवन\\V_VM_VNF निर्मीती\\JJ कार्य\\N_NN बंद\\RB करपाची\\V_VM_VNF शेतकार\\N_NN संघटणांत\\N_NN पैज\\N_NN लागल्या\\V_VM_VF .\\RD_PUNC']


In [18]:
print(len(testTaggedData))
ref_sentences = testTaggedData[:75]
print(len(ref_sentences))

1000
75


In [19]:
df_test = pd.DataFrame(ref_sentences, columns=['actual_tags'])
df_test.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 75 entries, 0 to 74
Data columns (total 1 columns):
 #   Column       Non-Null Count  Dtype 
---  ------       --------------  ----- 
 0   actual_tags  75 non-null     object
dtypes: object(1)
memory usage: 728.0+ bytes


In [20]:
df_test.tail()

,actual_tags
70,हांश्यास्पद\JJ स्थिती\N_NN अशी\DM_DMD आसा\V_VM...
71,"असले\PR_PRQ लोखंड\N_NN ,\RD_PUNC शिमीट\N_NN आन..."
72,सैमीक\JJ संकश्टांचेर\N_NN कोणाचोच\DM_DMI ताबो\...
73,पुरायपणान\N_NN सैमाचे\N_NN कृपेचेर\N_NN आदारीत...
74,पिकावळ\N_NN विम्याच्यो\N_NN खूबश्यो\QT_QTF येव...


In [21]:
def preprocess_sentence(sentence):
    words, tags = [], []
    for token in sentence.split():
        if "\\" in token:
            word, tag = token.rsplit("\\", 1)  # Split at last occurrence of '\'
            words.append(word)
            tags.append(tag)
    return words, tags

# Apply preprocessing
processed = df_test["actual_tags"].map(preprocess_sentence)

# Convert into two new columns
df_test[["sentence", "labels"]] = pd.DataFrame(processed.tolist(), index=df_test.index)

In [22]:
df_test.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 75 entries, 0 to 74
Data columns (total 3 columns):
 #   Column       Non-Null Count  Dtype 
---  ------       --------------  ----- 
 0   actual_tags  75 non-null     object
 1   sentence     75 non-null     object
 2   labels       75 non-null     object
dtypes: object(3)
memory usage: 1.9+ KB


In [23]:
def joinSen(words):
    sentence = " ".join(words)
    return sentence


df_test["joined_sent"] = df_test["sentence"].apply(joinSen)
df_test.head()

,actual_tags,sentence,labels,joined_sent
0,ग्रेटर\RD_UNK नोयडा\RD_UNK वेस्टाचे\RD_UNK त्र...,"[ग्रेटर, नोयडा, वेस्टाचे, त्रास, कमी, जावपाचें...","[RD_UNK, RD_UNK, RD_UNK, N_NN, QT_QTF, V_VM_VN...",ग्रेटर नोयडा वेस्टाचे त्रास कमी जावपाचें नांवू...
1,आपल्यो\PR_PRL मागण्यो\N_NN घेवन\V_VM_VNF निर्म...,"[आपल्यो, मागण्यो, घेवन, निर्मीती, कार्य, बंद, ...","[PR_PRL, N_NN, V_VM_VNF, JJ, N_NN, RB, V_VM_VN...",आपल्यो मागण्यो घेवन निर्मीती कार्य बंद करपाची ...
2,आयतरा\N_NNP मायचा\N_NN गांवांत\N_NN शेतकार\N_N...,"[आयतरा, मायचा, गांवांत, शेतकार, संघर्श, समितीन...","[N_NNP, N_NN, N_NN, N_NN, N_NN, N_NN, N_NN, V_...",आयतरा मायचा गांवांत शेतकार संघर्श समितीन पंचाय...
3,लुकसाण\N_NN भरपाय\N_NN दिवपाच्या\V_VM_VNF नांव...,"[लुकसाण, भरपाय, दिवपाच्या, नांवांचेर, शेतकारा,...","[N_NN, N_NN, V_VM_VNF, N_NN, N_NN, N_NN, N_NN,...",लुकसाण भरपाय दिवपाच्या नांवांचेर शेतकारा वांगड...
4,पंचायतींत\N_NN शेतकारांनी\N_NN सांगलां\V_VM_VN...,"[पंचायतींत, शेतकारांनी, सांगलां, की, म्हामंडळ,...","[N_NN, N_NN, V_VM_VNF, CC_CCS, N_NN, N_NN, N_N...",पंचायतींत शेतकारांनी सांगलां की म्हामंडळ शेतका...


In [24]:
df_test["output"] = df_test["joined_sent"].apply(predict_pos)
df_test.head()

,actual_tags,sentence,labels,joined_sent,output
0,ग्रेटर\RD_UNK नोयडा\RD_UNK वेस्टाचे\RD_UNK त्र...,"[ग्रेटर, नोयडा, वेस्टाचे, त्रास, कमी, जावपाचें...","[RD_UNK, RD_UNK, RD_UNK, N_NN, QT_QTF, V_VM_VN...",ग्रेटर नोयडा वेस्टाचे त्रास कमी जावपाचें नांवू...,"[(ग्रेटर, N_NN), (नोयडा, N_NNP), (वेस्टाचे, N_..."
1,आपल्यो\PR_PRL मागण्यो\N_NN घेवन\V_VM_VNF निर्म...,"[आपल्यो, मागण्यो, घेवन, निर्मीती, कार्य, बंद, ...","[PR_PRL, N_NN, V_VM_VNF, JJ, N_NN, RB, V_VM_VN...",आपल्यो मागण्यो घेवन निर्मीती कार्य बंद करपाची ...,"[(आपल्यो, PR_PRF), (मागण्यो, N_NN), (घेवन, V_V..."
2,आयतरा\N_NNP मायचा\N_NN गांवांत\N_NN शेतकार\N_N...,"[आयतरा, मायचा, गांवांत, शेतकार, संघर्श, समितीन...","[N_NNP, N_NN, N_NN, N_NN, N_NN, N_NN, N_NN, V_...",आयतरा मायचा गांवांत शेतकार संघर्श समितीन पंचाय...,"[(आयतरा, N_NNP), (मायचा, N_NN), (गांवांत, N_NN..."
3,लुकसाण\N_NN भरपाय\N_NN दिवपाच्या\V_VM_VNF नांव...,"[लुकसाण, भरपाय, दिवपाच्या, नांवांचेर, शेतकारा,...","[N_NN, N_NN, V_VM_VNF, N_NN, N_NN, N_NN, N_NN,...",लुकसाण भरपाय दिवपाच्या नांवांचेर शेतकारा वांगड...,"[(लुकसाण, N_NN), (भरपाय, N_NN), (दिवपाच्या, V_..."
4,पंचायतींत\N_NN शेतकारांनी\N_NN सांगलां\V_VM_VN...,"[पंचायतींत, शेतकारांनी, सांगलां, की, म्हामंडळ,...","[N_NN, N_NN, V_VM_VNF, CC_CCS, N_NN, N_NN, N_N...",पंचायतींत शेतकारांनी सांगलां की म्हामंडळ शेतका...,"[(पंचायतींत, N_NN), (शेतकारांनी, N_NN), (सांगल..."


In [25]:
import ast
def makeTagedSentences(data):
    try:
        data = str(data)  
        data_list = ast.literal_eval(data)  
        formatted_string = " ".join([f"{word}\\{tag}" for word, tag in data_list])
        return formatted_string
    except (ValueError, SyntaxError):
        return ""  

sent = "[('सहस्त्रधारा', 'N_NNP'), ('ही', 'DM_DMD'), ('देहरादूनाची', 'N_NNP'), ('पळोवपाची', 'V_VM_VNF'), ('मुखेल', 'JJ'), ('सुवात', 'N_NN'), ('.', 'RD_PUNC')]"
makeTagedSentences(sent)

'सहस्त्रधारा\\N_NNP ही\\DM_DMD देहरादूनाची\\N_NNP पळोवपाची\\V_VM_VNF मुखेल\\JJ सुवात\\N_NN .\\RD_PUNC'

In [26]:
df_test["joined_output"] = df_test["output"].apply(makeTagedSentences)
df_test.head()

,actual_tags,sentence,labels,joined_sent,output,joined_output
0,ग्रेटर\RD_UNK नोयडा\RD_UNK वेस्टाचे\RD_UNK त्र...,"[ग्रेटर, नोयडा, वेस्टाचे, त्रास, कमी, जावपाचें...","[RD_UNK, RD_UNK, RD_UNK, N_NN, QT_QTF, V_VM_VN...",ग्रेटर नोयडा वेस्टाचे त्रास कमी जावपाचें नांवू...,"[(ग्रेटर, N_NN), (नोयडा, N_NNP), (वेस्टाचे, N_...",ग्रेटर\N_NN नोयडा\N_NNP वेस्टाचे\N_NN त्रास\N_...
1,आपल्यो\PR_PRL मागण्यो\N_NN घेवन\V_VM_VNF निर्म...,"[आपल्यो, मागण्यो, घेवन, निर्मीती, कार्य, बंद, ...","[PR_PRL, N_NN, V_VM_VNF, JJ, N_NN, RB, V_VM_VN...",आपल्यो मागण्यो घेवन निर्मीती कार्य बंद करपाची ...,"[(आपल्यो, PR_PRF), (मागण्यो, N_NN), (घेवन, V_V...",आपल्यो\PR_PRF मागण्यो\N_NN घेवन\V_VM_VNF निर्म...
2,आयतरा\N_NNP मायचा\N_NN गांवांत\N_NN शेतकार\N_N...,"[आयतरा, मायचा, गांवांत, शेतकार, संघर्श, समितीन...","[N_NNP, N_NN, N_NN, N_NN, N_NN, N_NN, N_NN, V_...",आयतरा मायचा गांवांत शेतकार संघर्श समितीन पंचाय...,"[(आयतरा, N_NNP), (मायचा, N_NN), (गांवांत, N_NN...",आयतरा\N_NNP मायचा\N_NN गांवांत\N_NN शेतकार\N_N...
3,लुकसाण\N_NN भरपाय\N_NN दिवपाच्या\V_VM_VNF नांव...,"[लुकसाण, भरपाय, दिवपाच्या, नांवांचेर, शेतकारा,...","[N_NN, N_NN, V_VM_VNF, N_NN, N_NN, N_NN, N_NN,...",लुकसाण भरपाय दिवपाच्या नांवांचेर शेतकारा वांगड...,"[(लुकसाण, N_NN), (भरपाय, N_NN), (दिवपाच्या, V_...",लुकसाण\N_NN भरपाय\N_NN दिवपाच्या\V_VM_VNF नांव...
4,पंचायतींत\N_NN शेतकारांनी\N_NN सांगलां\V_VM_VN...,"[पंचायतींत, शेतकारांनी, सांगलां, की, म्हामंडळ,...","[N_NN, N_NN, V_VM_VNF, CC_CCS, N_NN, N_NN, N_N...",पंचायतींत शेतकारांनी सांगलां की म्हामंडळ शेतका...,"[(पंचायतींत, N_NN), (शेतकारांनी, N_NN), (सांगल...",पंचायतींत\N_NN शेतकारांनी\N_NN सांगलां\V_VM_VF...


In [27]:
df_test.drop(["output", "sentence", "labels"], axis=1, inplace=True)
df_test

,actual_tags,joined_sent,joined_output
0,ग्रेटर\RD_UNK नोयडा\RD_UNK वेस्टाचे\RD_UNK त्र...,ग्रेटर नोयडा वेस्टाचे त्रास कमी जावपाचें नांवू...,ग्रेटर\N_NN नोयडा\N_NNP वेस्टाचे\N_NN त्रास\N_...
1,आपल्यो\PR_PRL मागण्यो\N_NN घेवन\V_VM_VNF निर्म...,आपल्यो मागण्यो घेवन निर्मीती कार्य बंद करपाची ...,आपल्यो\PR_PRF मागण्यो\N_NN घेवन\V_VM_VNF निर्म...
2,आयतरा\N_NNP मायचा\N_NN गांवांत\N_NN शेतकार\N_N...,आयतरा मायचा गांवांत शेतकार संघर्श समितीन पंचाय...,आयतरा\N_NNP मायचा\N_NN गांवांत\N_NN शेतकार\N_N...
3,लुकसाण\N_NN भरपाय\N_NN दिवपाच्या\V_VM_VNF नांव...,लुकसाण भरपाय दिवपाच्या नांवांचेर शेतकारा वांगड...,लुकसाण\N_NN भरपाय\N_NN दिवपाच्या\V_VM_VNF नांव...
4,पंचायतींत\N_NN शेतकारांनी\N_NN सांगलां\V_VM_VN...,पंचायतींत शेतकारांनी सांगलां की म्हामंडळ शेतका...,पंचायतींत\N_NN शेतकारांनी\N_NN सांगलां\V_VM_VF...
...,...,...,...
70,हांश्यास्पद\JJ स्थिती\N_NN अशी\DM_DMD आसा\V_VM...,हांश्यास्पद स्थिती अशी आसा की आतां न्यायालयाक ...,हांश्यास्पद\N_NN स्थिती\N_NN अशी\DM_DMD आसा\V_...
71,"असले\PR_PRQ लोखंड\N_NN ,\RD_PUNC शिमीट\N_NN आन...","असले लोखंड , शिमीट आनी दुसरें उद्येगीक उत्पादन...","असले\DM_DMD लोखंड\N_NN ,\RD_PUNC शिमीट\N_NN आन..."
72,सैमीक\JJ संकश्टांचेर\N_NN कोणाचोच\DM_DMI ताबो\...,"सैमीक संकश्टांचेर कोणाचोच ताबो ना , पूण असलीं ...",सैमीक\JJ संकश्टांचेर\N_NN कोणाचोच\PR_PRQ ताबो\...
73,पुरायपणान\N_NN सैमाचे\N_NN कृपेचेर\N_NN आदारीत...,पुरायपणान सैमाचे कृपेचेर आदारीत शेतकार्‍याच्या...,पुरायपणान\N_NN सैमाचे\N_NN कृपेचेर\N_NN आदारीत...


In [28]:
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, classification_report

def evaluate_tagging(ref_sent, predicted_sent):
    #print("ref_sent: ", ref_sent)
    #print("predicted_sent: ", predicted_sent)
    # Handle missing values
    if not isinstance(ref_sent, str) or not isinstance(predicted_sent, str):
        return 0.0, 0.0, 0.0, 0.0  # Return default scores for missing values

    # Split sentences into word-tag pairs
    ref_tags = ref_sent.split()
    pred_tags = predicted_sent.split()

    if len(ref_tags) != len(pred_tags):
        return 0.0, 0.0, 0.0, 0.0
    

    # Extract only the tags from word\tag pairs
    ref_labels = [token.split("\\")[-1] for token in ref_tags]
    pred_labels = [token.split("\\")[-1] for token in pred_tags]

    #print(len(ref_labels), len(pred_labels))

    # Compute accuracy
    accuracy = accuracy_score(ref_labels, pred_labels)

    # Compute precision, recall, and F1-score (weighted for class imbalance)
    precision, recall, f1, _ = precision_recall_fscore_support(ref_labels, pred_labels, average='weighted', zero_division=0)

    return accuracy, precision, recall, f1, ref_labels, pred_labels

In [29]:
def add_length_columns(df):
    # Compute lengths of actual_tags and predicted_sentences
    df['ref_sentences_length'] = df['actual_tags'].apply(lambda x: len(str(x).split()))
    df['joined_sent_length'] = df['joined_output'].apply(lambda x: len(str(x).split()))
    
    # Find row indices where lengths do not match
    mismatched_rows = df[df['ref_sentences_length'] != df['joined_sent_length']].index.tolist()
    
    # Remove mismatched rows from the DataFrame
    #df = df.drop(mismatched_rows)
    
    return df, mismatched_rows

# Example usage:
df_exp, mismatched_indices = add_length_columns(df_test)
for ind in mismatched_indices:
    df_test = df_test.drop(ind)

df_test.info()

<class 'pandas.core.frame.DataFrame'>
Int64Index: 74 entries, 0 to 74
Data columns (total 5 columns):
 #   Column                Non-Null Count  Dtype 
---  ------                --------------  ----- 
 0   actual_tags           74 non-null     object
 1   joined_sent           74 non-null     object
 2   joined_output         74 non-null     object
 3   ref_sentences_length  74 non-null     int64 
 4   joined_sent_length    74 non-null     int64 
dtypes: int64(2), object(3)
memory usage: 3.5+ KB


In [30]:
# Apply function to each row
df_test[["Accuracy", 
        "Precision", 
        "Recall", 
        "F1 Score", 
        "Ref Labels", 
        "Pred Labels"]] = df_test.apply(lambda row: evaluate_tagging(row["actual_tags"], row["joined_output"]), 
                                       axis=1, 
                                       result_type="expand")
df_test.head()

,actual_tags,joined_sent,joined_output,ref_sentences_length,joined_sent_length,Accuracy,Precision,Recall,F1 Score,Ref Labels,Pred Labels
0,ग्रेटर\RD_UNK नोयडा\RD_UNK वेस्टाचे\RD_UNK त्र...,ग्रेटर नोयडा वेस्टाचे त्रास कमी जावपाचें नांवू...,ग्रेटर\N_NN नोयडा\N_NNP वेस्टाचे\N_NN त्रास\N_...,9,9,0.666667,0.555556,0.666667,0.592593,"[RD_UNK, RD_UNK, RD_UNK, N_NN, QT_QTF, V_VM_VN...","[N_NN, N_NNP, N_NN, N_NN, QT_QTF, V_VM_VNF, N_..."
1,आपल्यो\PR_PRL मागण्यो\N_NN घेवन\V_VM_VNF निर्म...,आपल्यो मागण्यो घेवन निर्मीती कार्य बंद करपाची ...,आपल्यो\PR_PRF मागण्यो\N_NN घेवन\V_VM_VNF निर्म...,12,12,0.750000,0.680556,0.750000,0.712121,"[PR_PRL, N_NN, V_VM_VNF, JJ, N_NN, RB, V_VM_VN...","[PR_PRF, N_NN, V_VM_VNF, N_NN, N_NN, RB, V_VM_..."
2,आयतरा\N_NNP मायचा\N_NN गांवांत\N_NN शेतकार\N_N...,आयतरा मायचा गांवांत शेतकार संघर्श समितीन पंचाय...,आयतरा\N_NNP मायचा\N_NN गांवांत\N_NN शेतकार\N_N...,17,17,0.823529,0.835294,0.823529,0.818369,"[N_NNP, N_NN, N_NN, N_NN, N_NN, N_NN, N_NN, V_...","[N_NNP, N_NN, N_NN, N_NN, N_NN, N_NN, N_NN, V_..."
3,लुकसाण\N_NN भरपाय\N_NN दिवपाच्या\V_VM_VNF नांव...,लुकसाण भरपाय दिवपाच्या नांवांचेर शेतकारा वांगड...,लुकसाण\N_NN भरपाय\N_NN दिवपाच्या\V_VM_VNF नांव...,21,21,0.857143,0.857143,0.857143,0.857143,"[N_NN, N_NN, V_VM_VNF, N_NN, N_NN, N_NN, N_NN,...","[N_NN, N_NN, V_VM_VNF, N_NN, N_NN, PSP, N_NN, ..."
4,पंचायतींत\N_NN शेतकारांनी\N_NN सांगलां\V_VM_VN...,पंचायतींत शेतकारांनी सांगलां की म्हामंडळ शेतका...,पंचायतींत\N_NN शेतकारांनी\N_NN सांगलां\V_VM_VF...,10,10,0.800000,0.800000,0.800000,0.800000,"[N_NN, N_NN, V_VM_VNF, CC_CCS, N_NN, N_NN, N_N...","[N_NN, N_NN, V_VM_VF, CC_CCS, N_NN, N_NN, N_NN..."


In [31]:
df_test.to_excel("test_svm_75_output.xlsx", index=False)

In [32]:
# Flatten all reference and predicted labels for classification report
all_ref_labels = [label for labels in df_test["Ref Labels"] for label in labels]
all_pred_labels = [label for labels in df_test["Pred Labels"] for label in labels]

# Generate classification report
print("\nClassification Report:\n")
print(classification_report(all_ref_labels, all_pred_labels, zero_division=0))


Classification Report:

              precision    recall  f1-score   support

      CC_CCD       1.00      0.96      0.98        28
      CC_CCS       0.94      0.85      0.89        20
      DM_DMD       0.94      0.81      0.87        37
      DM_DMI       0.00      0.00      0.00         1
      DM_DMR       0.40      1.00      0.57         2
          JJ       0.61      0.53      0.56        87
        N_NN       0.83      0.93      0.88       407
       N_NNP       0.25      0.50      0.33        10
       N_NST       0.65      0.83      0.73        18
      PR_PRF       0.83      1.00      0.91         5
      PR_PRI       0.00      0.00      0.00         1
      PR_PRL       1.00      0.50      0.67         6
      PR_PRP       0.25      0.67      0.36         3
      PR_PRQ       0.00      0.00      0.00         1
         PSP       0.94      0.79      0.86        39
      QT_QTC       1.00      0.89      0.94        38
      QT_QTF       0.88      0.88      0.88        17
  

In [33]:
import pandas as pd
df_ref = pd.read_excel('ref_sentences/ref_sentences_Mannual.xlsx', engine='openpyxl')
df_ref.head()

,sentences,ref_sentences,Unnamed: 2,Unnamed: 3,Unnamed: 4,Unnamed: 5,Unnamed: 6,Unnamed: 7,Unnamed: 8,Unnamed: 9,...,Unnamed: 15,Unnamed: 16,Unnamed: 17,Unnamed: 18,Unnamed: 19,Unnamed: 20,Unnamed: 21,Unnamed: 22,Unnamed: 23,Unnamed: 24
0,मनशाच्या शरिरांत 206 हाडां आसात .,मनशाच्या\N_NN शरिरांत\N_NN 206\QT_QTC हाडां\N_...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,मनशाचें काळीज दिसाक सुमार 1 लाख फावटीं ठोके दि...,मनशाचें\N_NN काळीज\N_NN दिसाक\N_NN सुमार\RB 1\...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,सूर्य पृथ्वी परस 3 लाख परस चड पटीन आकारांत व्ह...,सूर्य\N_NNP पृथ्वी\N_NNP परस\PSP 3\QT_QTC लाख\...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,नील आर्मस्ट्राँग हो 1969 वर्सा चंद्राचेर पावल ...,नील\N_NNP आर्मस्ट्राँग\N_NNP हो\PR_PRP 1969\N_...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,पृथ्वीचो सगल्यांत व्हडलो आनी एकमेव सैमीक उपगिर...,पृथ्वीचो\N_NNP सगल्यांत\RP_INTF व्हडलो\JJ आनी\...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [34]:
df_ref = df_ref[["sentences","ref_sentences"]]
df_ref.head()

,sentences,ref_sentences
0,मनशाच्या शरिरांत 206 हाडां आसात .,मनशाच्या\N_NN शरिरांत\N_NN 206\QT_QTC हाडां\N_...
1,मनशाचें काळीज दिसाक सुमार 1 लाख फावटीं ठोके दि...,मनशाचें\N_NN काळीज\N_NN दिसाक\N_NN सुमार\RB 1\...
2,सूर्य पृथ्वी परस 3 लाख परस चड पटीन आकारांत व्ह...,सूर्य\N_NNP पृथ्वी\N_NNP परस\PSP 3\QT_QTC लाख\...
3,नील आर्मस्ट्राँग हो 1969 वर्सा चंद्राचेर पावल ...,नील\N_NNP आर्मस्ट्राँग\N_NNP हो\PR_PRP 1969\N_...
4,पृथ्वीचो सगल्यांत व्हडलो आनी एकमेव सैमीक उपगिर...,पृथ्वीचो\N_NNP सगल्यांत\RP_INTF व्हडलो\JJ आनी\...


In [35]:
df_ref = df_ref.iloc[:100]

In [36]:
df_ref.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 100 entries, 0 to 99
Data columns (total 2 columns):
 #   Column         Non-Null Count  Dtype 
---  ------         --------------  ----- 
 0   sentences      100 non-null    object
 1   ref_sentences  100 non-null    object
dtypes: object(2)
memory usage: 1.7+ KB


In [37]:
df_ref["predicted_sentences"] = df_ref["sentences"].apply(predict_pos)
df_ref.head()

,sentences,ref_sentences,predicted_sentences
0,मनशाच्या शरिरांत 206 हाडां आसात .,मनशाच्या\N_NN शरिरांत\N_NN 206\QT_QTC हाडां\N_...,"[(मनशाच्या, N_NN), (शरिरांत, N_NN), (206, QT_Q..."
1,मनशाचें काळीज दिसाक सुमार 1 लाख फावटीं ठोके दि...,मनशाचें\N_NN काळीज\N_NN दिसाक\N_NN सुमार\RB 1\...,"[(मनशाचें, N_NN), (काळीज, N_NN), (दिसाक, N_NN)..."
2,सूर्य पृथ्वी परस 3 लाख परस चड पटीन आकारांत व्ह...,सूर्य\N_NNP पृथ्वी\N_NNP परस\PSP 3\QT_QTC लाख\...,"[(सूर्य, N_NN), (पृथ्वी, N_NN), (परस, PSP), (3..."
3,नील आर्मस्ट्राँग हो 1969 वर्सा चंद्राचेर पावल ...,नील\N_NNP आर्मस्ट्राँग\N_NNP हो\PR_PRP 1969\N_...,"[(नील, N_NNP), (आर्मस्ट्राँग, N_NNP), (हो, DM_..."
4,पृथ्वीचो सगल्यांत व्हडलो आनी एकमेव सैमीक उपगिर...,पृथ्वीचो\N_NNP सगल्यांत\RP_INTF व्हडलो\JJ आनी\...,"[(पृथ्वीचो, N_NN), (सगल्यांत, RP_INTF), (व्हडल..."


In [38]:
import ast
def makeTagedSentences(data):
    try:
        data = str(data)  
        data_list = ast.literal_eval(data)  
        formatted_string = " ".join([f"{word}\\{tag}" for word, tag in data_list])
        return formatted_string
    except (ValueError, SyntaxError):
        return ""  

sent = "[('सहस्त्रधारा', 'N_NNP'), ('ही', 'DM_DMD'), ('देहरादूनाची', 'N_NNP'), ('पळोवपाची', 'V_VM_VNF'), ('मुखेल', 'JJ'), ('सुवात', 'N_NN'), ('.', 'RD_PUNC')]"
makeTagedSentences(sent)

'सहस्त्रधारा\\N_NNP ही\\DM_DMD देहरादूनाची\\N_NNP पळोवपाची\\V_VM_VNF मुखेल\\JJ सुवात\\N_NN .\\RD_PUNC'

In [39]:
df_ref["predicted_sentences_joined"] = df_ref["predicted_sentences"].apply(makeTagedSentences)
df_ref

,sentences,ref_sentences,predicted_sentences,predicted_sentences_joined
0,मनशाच्या शरिरांत 206 हाडां आसात .,मनशाच्या\N_NN शरिरांत\N_NN 206\QT_QTC हाडां\N_...,"[(मनशाच्या, N_NN), (शरिरांत, N_NN), (206, QT_Q...",मनशाच्या\N_NN शरिरांत\N_NN 206\QT_QTC हाडां\N_...
1,मनशाचें काळीज दिसाक सुमार 1 लाख फावटीं ठोके दि...,मनशाचें\N_NN काळीज\N_NN दिसाक\N_NN सुमार\RB 1\...,"[(मनशाचें, N_NN), (काळीज, N_NN), (दिसाक, N_NN)...",मनशाचें\N_NN काळीज\N_NN दिसाक\N_NN सुमार\QT_QT...
2,सूर्य पृथ्वी परस 3 लाख परस चड पटीन आकारांत व्ह...,सूर्य\N_NNP पृथ्वी\N_NNP परस\PSP 3\QT_QTC लाख\...,"[(सूर्य, N_NN), (पृथ्वी, N_NN), (परस, PSP), (3...",सूर्य\N_NN पृथ्वी\N_NN परस\PSP 3\QT_QTC लाख\N_...
3,नील आर्मस्ट्राँग हो 1969 वर्सा चंद्राचेर पावल ...,नील\N_NNP आर्मस्ट्राँग\N_NNP हो\PR_PRP 1969\N_...,"[(नील, N_NNP), (आर्मस्ट्राँग, N_NNP), (हो, DM_...",नील\N_NNP आर्मस्ट्राँग\N_NNP हो\DM_DMD 1969\QT...
4,पृथ्वीचो सगल्यांत व्हडलो आनी एकमेव सैमीक उपगिर...,पृथ्वीचो\N_NNP सगल्यांत\RP_INTF व्हडलो\JJ आनी\...,"[(पृथ्वीचो, N_NN), (सगल्यांत, RP_INTF), (व्हडल...",पृथ्वीचो\N_NN सगल्यांत\RP_INTF व्हडलो\JJ आनी\C...
...,...,...,...,...
95,रातभर तातूंत पोंयेंतल्यान उदक गळटा आनी सकाळ मे...,रातभर\N_NN तातूंत\PR_PRL पोंयेंतल्यान\N_NN उदक...,"[(रातभर, N_NN), (तातूंत, DM_DMR), (पोंयेंतल्या...",रातभर\N_NN तातूंत\DM_DMR पोंयेंतल्यान\N_NN उदक...
96,सकाळीं रेंदेर माडाचेर चडटात आनी माडाक बांदिल्ल...,सकाळीं\N_NN रेंदेर\N_NN माडाचेर\N_NN चडटात\V_V...,"[(सकाळीं, N_NN), (रेंदेर, N_NN), (माडाचेर, N_N...",सकाळीं\N_NN रेंदेर\N_NN माडाचेर\N_NN चडटात\V_V...
97,तातूंत सूर एकठांय जाल्लो आसता .,तातूंत\PR_PRL सूर\N_NN एकठांय\RB जाल्लो\V_VM_V...,"[(तातूंत, DM_DMR), (सूर, N_NN), (एकठांय, RB), ...",तातूंत\DM_DMR सूर\N_NN एकठांय\RB जाल्लो\V_VM_V...
98,माडाची सूर पितात लेगीत .,माडाची\N_NN सूर\N_NN पितात\V_VM_VF लेगीत\PSP ....,"[(माडाची, N_NN), (सूर, N_NN), (पितात, V_VM_VF)...",माडाची\N_NN सूर\N_NN पितात\V_VM_VF लेगीत\PSP ....


In [40]:
print(df_ref["ref_sentences"][0])
print(df_ref["predicted_sentences_joined"][0])

मनशाच्या\N_NN शरिरांत\N_NN 206\QT_QTC हाडां\N_NN आसात\V_VM_VF .\RD_PUNC
मनशाच्या\N_NN शरिरांत\N_NN 206\QT_QTC हाडां\N_NN आसात\V_VM_VF .\RD_PUNC


In [41]:
def add_length_columns(df):
    # Compute lengths of actual_tags and predicted_sentences
    df['actual_tag_length'] = df['ref_sentences'].apply(lambda x: len(str(x).split()))
    df['pred_tag_length'] = df['predicted_sentences_joined'].apply(lambda x: len(str(x).split()))
    
    # Find row indices where lengths do not match
    mismatched_rows = df[df['actual_tag_length'] != df['pred_tag_length']].index.tolist()
    
    return df, mismatched_rows

# Example usage:
df_exp, mismatched_indices = add_length_columns(df_ref)

mismatched_indices

[43, 55, 56, 75, 80, 92, 99]

In [42]:
for ind in mismatched_indices:
    df_ref = df_ref.drop(ind)

In [43]:
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, classification_report

def evaluate_tagging(ref_sent, predicted_sent):
    #print("ref_sent: ", ref_sent)
    #print("predicted_sent: ", predicted_sent)
    # Handle missing values
    if not isinstance(ref_sent, str) or not isinstance(predicted_sent, str):
        return 0.0, 0.0, 0.0, 0.0  # Return default scores for missing values

    # Split sentences into word-tag pairs
    ref_tags = ref_sent.split()
    pred_tags = predicted_sent.split()

    if len(ref_tags) != len(pred_tags):
        return 0.0, 0.0, 0.0, 0.0
    

    # Extract only the tags from word\tag pairs
    ref_labels = [token.split("\\")[-1] for token in ref_tags]
    pred_labels = [token.split("\\")[-1] for token in pred_tags]

    #print(len(ref_labels), len(pred_labels))

    # Compute accuracy
    accuracy = accuracy_score(ref_labels, pred_labels)

    # Compute precision, recall, and F1-score (weighted for class imbalance)
    precision, recall, f1, _ = precision_recall_fscore_support(ref_labels, pred_labels, average='weighted', zero_division=0)

    return accuracy, precision, recall, f1, ref_labels, pred_labels

In [44]:
# Apply function to each row
df_ref[["Accuracy", 
        "Precision", 
        "Recall", 
        "F1 Score", 
        "Ref Labels", 
        "Pred Labels"]] = df_ref.apply(lambda row: evaluate_tagging(row["ref_sentences"], row["predicted_sentences_joined"]), 
                                       axis=1, 
                                       result_type="expand")
df_ref.head()

,sentences,ref_sentences,predicted_sentences,predicted_sentences_joined,actual_tag_length,pred_tag_length,Accuracy,Precision,Recall,F1 Score,Ref Labels,Pred Labels
0,मनशाच्या शरिरांत 206 हाडां आसात .,मनशाच्या\N_NN शरिरांत\N_NN 206\QT_QTC हाडां\N_...,"[(मनशाच्या, N_NN), (शरिरांत, N_NN), (206, QT_Q...",मनशाच्या\N_NN शरिरांत\N_NN 206\QT_QTC हाडां\N_...,6,6,1.000000,1.000000,1.000000,1.000000,"[N_NN, N_NN, QT_QTC, N_NN, V_VM_VF, RD_PUNC]","[N_NN, N_NN, QT_QTC, N_NN, V_VM_VF, RD_PUNC]"
1,मनशाचें काळीज दिसाक सुमार 1 लाख फावटीं ठोके दि...,मनशाचें\N_NN काळीज\N_NN दिसाक\N_NN सुमार\RB 1\...,"[(मनशाचें, N_NN), (काळीज, N_NN), (दिसाक, N_NN)...",मनशाचें\N_NN काळीज\N_NN दिसाक\N_NN सुमार\QT_QT...,10,10,0.900000,0.900000,0.900000,0.900000,"[N_NN, N_NN, N_NN, RB, QT_QTC, N_NN, N_NN, N_N...","[N_NN, N_NN, N_NN, QT_QTF, QT_QTC, N_NN, N_NN,..."
2,सूर्य पृथ्वी परस 3 लाख परस चड पटीन आकारांत व्ह...,सूर्य\N_NNP पृथ्वी\N_NNP परस\PSP 3\QT_QTC लाख\...,"[(सूर्य, N_NN), (पृथ्वी, N_NN), (परस, PSP), (3...",सूर्य\N_NN पृथ्वी\N_NN परस\PSP 3\QT_QTC लाख\N_...,20,20,0.850000,0.756250,0.850000,0.792308,"[N_NNP, N_NNP, PSP, QT_QTC, N_NN, PSP, QT_QTF,...","[N_NN, N_NN, PSP, QT_QTC, N_NN, PSP, QT_QTF, N..."
3,नील आर्मस्ट्राँग हो 1969 वर्सा चंद्राचेर पावल ...,नील\N_NNP आर्मस्ट्राँग\N_NNP हो\PR_PRP 1969\N_...,"[(नील, N_NNP), (आर्मस्ट्राँग, N_NNP), (हो, DM_...",नील\N_NNP आर्मस्ट्राँग\N_NNP हो\DM_DMD 1969\QT...,12,12,0.833333,0.916667,0.833333,0.869048,"[N_NNP, N_NNP, PR_PRP, N_NN, N_NN, N_NNP, N_NN...","[N_NNP, N_NNP, DM_DMD, QT_QTC, N_NN, N_NNP, N_..."
4,पृथ्वीचो सगल्यांत व्हडलो आनी एकमेव सैमीक उपगिर...,पृथ्वीचो\N_NNP सगल्यांत\RP_INTF व्हडलो\JJ आनी\...,"[(पृथ्वीचो, N_NN), (सगल्यांत, RP_INTF), (व्हडल...",पृथ्वीचो\N_NN सगल्यांत\RP_INTF व्हडलो\JJ आनी\C...,10,10,0.900000,0.950000,0.900000,0.900000,"[N_NNP, RP_INTF, JJ, CC_CCD, JJ, JJ, N_NN, CC_...","[N_NN, RP_INTF, JJ, CC_CCD, JJ, JJ, N_NN, CC_C..."


In [45]:
# Flatten all reference and predicted labels for classification report
all_ref_labels = [label for labels in df_ref["Ref Labels"] for label in labels]
all_pred_labels = [label for labels in df_ref["Pred Labels"] for label in labels]

# Generate classification report
print("\nClassification Report:\n")
print(classification_report(all_ref_labels, all_pred_labels, zero_division=0))


Classification Report:

              precision    recall  f1-score   support

      CC_CCD       1.00      1.00      1.00        20
      CC_CCS       0.95      0.95      0.95        20
      DM_DMD       0.94      1.00      0.97        30
      DM_DMI       1.00      1.00      1.00         1
      DM_DMR       0.33      0.50      0.40         2
          JJ       1.00      0.98      0.99        60
        N_NN       0.96      0.93      0.95       345
       N_NNP       0.75      0.85      0.80        62
       N_NST       0.82      1.00      0.90        18
      PR_PRF       1.00      1.00      1.00         6
      PR_PRL       0.00      0.00      0.00         2
      PR_PRP       1.00      0.88      0.93         8
         PSP       1.00      0.84      0.92        32
      QT_QTC       0.97      1.00      0.98        28
      QT_QTF       0.87      1.00      0.93        20
      QT_QTO       1.00      1.00      1.00         3
          RB       1.00      0.92      0.96        13
  

In [46]:
df_ref.to_excel("test_svm_100_mannual.xlsx", index=False)

In [59]:


df_anwani = pd.read_excel('ref_sentences/anwani_ref_sentences.xlsx', engine='openpyxl')
df_anwani.head()

,sentences_cleaned,predicted_sentences,tagged_sentences,Unnamed: 3,Unnamed: 4,Unnamed: 5,Unnamed: 6,Unnamed: 7,Unnamed: 8,Unnamed: 9,...,Unnamed: 16,Unnamed: 17,Unnamed: 18,Unnamed: 19,Unnamed: 20,Unnamed: 21,Unnamed: 22,Unnamed: 23,Unnamed: 24,Unnamed: 25
0,आयकलां ? लागीं ना आमचो गांव . चलून चलून कितलीं...,आयकलां\V_VM_VF ?\RD_PUNC लागीं\N_NST ना\RP_NEG...,आयकलां\V_VM_VF ?\RD_PUNC लागीं\N_NST ना\RP_NEG...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,दोन तीन दिसांनी जायत सुरू येरादारी . इतली कित्...,दोन\QT_QTC तीन\QT_QTC दिसांनी\N_NN जायत\V_VAUX...,दोन\QT_QTC तीन\QT_QTC दिसांनी\N_NN जायत\V_VM_V...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,आनी जाली ना जाल्यार ? कोण घेतलो आमकां घरांत ? ...,आनी\CC_CCD जाली\V_VM_VF ना\RP_NEG जाल्यार\CC_C...,आनी\CC_CCD जाली\V_VM_VNF ना\RP_NEG जाल्यार\CC_...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,"हांव अशें आसा म्हणून तर सांगतां , घाई नाका . आ...",हांव\PR_PRP अशें\DM_DMD आसा\V_VM_VF म्हणून\CC_...,हांव\PR_PRP अशें\DM_DMD आसा\V_VM_VF म्हणून\CC_...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,पूण हांगा रावनय जावचें ना . चलत रावल्यार मेळत ...,पूण\CC_CCS हांगा\N_NST रावनय\V_VM_VNF जावचें\V...,पूण\CC_CCS हांगा\N_NST रावनय\V_VM_VNF जावचें\V...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [60]:
df_anwani = df_anwani.iloc[:, :3]
df_anwani.head()

,sentences_cleaned,predicted_sentences,tagged_sentences
0,आयकलां ? लागीं ना आमचो गांव . चलून चलून कितलीं...,आयकलां\V_VM_VF ?\RD_PUNC लागीं\N_NST ना\RP_NEG...,आयकलां\V_VM_VF ?\RD_PUNC लागीं\N_NST ना\RP_NEG...
1,दोन तीन दिसांनी जायत सुरू येरादारी . इतली कित्...,दोन\QT_QTC तीन\QT_QTC दिसांनी\N_NN जायत\V_VAUX...,दोन\QT_QTC तीन\QT_QTC दिसांनी\N_NN जायत\V_VM_V...
2,आनी जाली ना जाल्यार ? कोण घेतलो आमकां घरांत ? ...,आनी\CC_CCD जाली\V_VM_VF ना\RP_NEG जाल्यार\CC_C...,आनी\CC_CCD जाली\V_VM_VNF ना\RP_NEG जाल्यार\CC_...
3,"हांव अशें आसा म्हणून तर सांगतां , घाई नाका . आ...",हांव\PR_PRP अशें\DM_DMD आसा\V_VM_VF म्हणून\CC_...,हांव\PR_PRP अशें\DM_DMD आसा\V_VM_VF म्हणून\CC_...
4,पूण हांगा रावनय जावचें ना . चलत रावल्यार मेळत ...,पूण\CC_CCS हांगा\N_NST रावनय\V_VM_VNF जावचें\V...,पूण\CC_CCS हांगा\N_NST रावनय\V_VM_VNF जावचें\V...


In [61]:
df_anwani = df_anwani.iloc[:100, :3]
df_anwani.tail()

,sentences_cleaned,predicted_sentences,tagged_sentences
95,बस . लागीं लागीं कोण आसा अशें दिसना .,बस\N_NN .\RD_PUNC लागीं\N_NST लागीं\N_NST कोण\...,बस\V_VM_VF .\RD_PUNC लागीं\N_NST लागीं\N_NST क...
96,सायबान आमकां आनीक थोडे दीस कामार दवरल्यार बरें...,सायबान\N_NN आमकां\PR_PRP आनीक\QT_QTF थोडे\QT_Q...,सायबान\N_NN आमकां\PR_PRP आनीक\RP_INTF थोडे\QT_...
97,काम करना आसतना किद्याक पोंसता आशिल्लो तो ?,काम\N_NN करना\N_NNP आसतना\V_VM_VNF किद्याक\PR_...,काम\N_NN करना\V_VM_VNF आसतना\V_VM_VNF किद्याक\...
98,तांचें बरें न्ही ? तांकां खंयच चलत वच्चें पडले...,तांचें\PR_PRP बरें\RB न्ही\RP_NEG ?\RD_PUNC ता...,तांचें\PR_PRP बरें\RB न्ही\RP_NEG ?\RD_PUNC ता...
99,तांचें किदें बाये ? आमचे सारकी काटकसरीची जीण न...,तांचें\PR_PRP किदें\PR_PRQ बाये\N_NN ?\RD_PUNC...,तांचें\PR_PRP किदें\PR_PRQ बाये\N_NN ?\RD_PUNC...


In [62]:
df_anwani["output"] = df_anwani["sentences_cleaned"].apply(predict_pos)
df_anwani.head()

,sentences_cleaned,predicted_sentences,tagged_sentences,output
0,आयकलां ? लागीं ना आमचो गांव . चलून चलून कितलीं...,आयकलां\V_VM_VF ?\RD_PUNC लागीं\N_NST ना\RP_NEG...,आयकलां\V_VM_VF ?\RD_PUNC लागीं\N_NST ना\RP_NEG...,"[(आयकलां, V_VM_VF), (?, RD_PUNC), (लागीं, N_NS..."
1,दोन तीन दिसांनी जायत सुरू येरादारी . इतली कित्...,दोन\QT_QTC तीन\QT_QTC दिसांनी\N_NN जायत\V_VAUX...,दोन\QT_QTC तीन\QT_QTC दिसांनी\N_NN जायत\V_VM_V...,"[(दोन, QT_QTC), (तीन, QT_QTC), (दिसांनी, N_NN)..."
2,आनी जाली ना जाल्यार ? कोण घेतलो आमकां घरांत ? ...,आनी\CC_CCD जाली\V_VM_VF ना\RP_NEG जाल्यार\CC_C...,आनी\CC_CCD जाली\V_VM_VNF ना\RP_NEG जाल्यार\CC_...,"[(आनी, CC_CCD), (जाली, V_VM_VF), (ना, RP_NEG),..."
3,"हांव अशें आसा म्हणून तर सांगतां , घाई नाका . आ...",हांव\PR_PRP अशें\DM_DMD आसा\V_VM_VF म्हणून\CC_...,हांव\PR_PRP अशें\DM_DMD आसा\V_VM_VF म्हणून\CC_...,"[(हांव, PR_PRP), (अशें, DM_DMD), (आसा, V_VM_VF..."
4,पूण हांगा रावनय जावचें ना . चलत रावल्यार मेळत ...,पूण\CC_CCS हांगा\N_NST रावनय\V_VM_VNF जावचें\V...,पूण\CC_CCS हांगा\N_NST रावनय\V_VM_VNF जावचें\V...,"[(पूण, CC_CCS), (हांगा, N_NST), (रावनय, N_NN),..."


In [63]:
df_anwani["joined_output"] = df_anwani["output"].apply(makeTagedSentences)
df_anwani.drop(["output"], axis=1, inplace=True)
df_anwani.drop(["predicted_sentences"], axis=1, inplace=True)
df_anwani.head()

,sentences_cleaned,tagged_sentences,joined_output
0,आयकलां ? लागीं ना आमचो गांव . चलून चलून कितलीं...,आयकलां\V_VM_VF ?\RD_PUNC लागीं\N_NST ना\RP_NEG...,आयकलां\V_VM_VF ?\RD_PUNC लागीं\N_NST ना\RP_NEG...
1,दोन तीन दिसांनी जायत सुरू येरादारी . इतली कित्...,दोन\QT_QTC तीन\QT_QTC दिसांनी\N_NN जायत\V_VM_V...,दोन\QT_QTC तीन\QT_QTC दिसांनी\N_NN जायत\V_VM_V...
2,आनी जाली ना जाल्यार ? कोण घेतलो आमकां घरांत ? ...,आनी\CC_CCD जाली\V_VM_VNF ना\RP_NEG जाल्यार\CC_...,आनी\CC_CCD जाली\V_VM_VF ना\RP_NEG जाल्यार\CC_C...
3,"हांव अशें आसा म्हणून तर सांगतां , घाई नाका . आ...",हांव\PR_PRP अशें\DM_DMD आसा\V_VM_VF म्हणून\CC_...,हांव\PR_PRP अशें\DM_DMD आसा\V_VM_VF म्हणून\V_V...
4,पूण हांगा रावनय जावचें ना . चलत रावल्यार मेळत ...,पूण\CC_CCS हांगा\N_NST रावनय\V_VM_VNF जावचें\V...,पूण\CC_CCS हांगा\N_NST रावनय\N_NN जावचें\V_VM_...


In [64]:
import pandas as pd

def add_length_columns(df):
    # Compute lengths of actual_tags and predicted_sentences
    df['ref_sentences_length'] = df['tagged_sentences'].apply(lambda x: len(str(x).split()))
    df['joined_sent_length'] = df['joined_output'].apply(lambda x: len(str(x).split()))
    
    # Find row indices where lengths do not match
    mismatched_rows = df[df['ref_sentences_length'] != df['joined_sent_length']].index.tolist()
    
    # Remove mismatched rows from the DataFrame
    #df = df.drop(mismatched_rows)
    
    return df, mismatched_rows

# Example usage:
df_anwani, mismatched_indices = add_length_columns(df_anwani)
mismatched_indices

[48, 49]

In [65]:
for ind in mismatched_indices:
    df_anwani = df_anwani.drop(ind)

df_anwani.info()

<class 'pandas.core.frame.DataFrame'>
Int64Index: 98 entries, 0 to 99
Data columns (total 5 columns):
 #   Column                Non-Null Count  Dtype 
---  ------                --------------  ----- 
 0   sentences_cleaned     98 non-null     object
 1   tagged_sentences      98 non-null     object
 2   joined_output         98 non-null     object
 3   ref_sentences_length  98 non-null     int64 
 4   joined_sent_length    98 non-null     int64 
dtypes: int64(2), object(3)
memory usage: 4.6+ KB


In [66]:
# Apply function to each row
df_anwani[["Accuracy", 
        "Precision", 
        "Recall", 
        "F1 Score", 
        "Ref Labels", 
        "Pred Labels"]] = df_anwani.apply(lambda row: evaluate_tagging(row["tagged_sentences"], row["joined_output"]), 
                                       axis=1, 
                                       result_type="expand")
df_anwani.head()

,sentences_cleaned,tagged_sentences,joined_output,ref_sentences_length,joined_sent_length,Accuracy,Precision,Recall,F1 Score,Ref Labels,Pred Labels
0,आयकलां ? लागीं ना आमचो गांव . चलून चलून कितलीं...,आयकलां\V_VM_VF ?\RD_PUNC लागीं\N_NST ना\RP_NEG...,आयकलां\V_VM_VF ?\RD_PUNC लागीं\N_NST ना\RP_NEG...,14,14,0.785714,0.714286,0.785714,0.738095,"[V_VM_VF, RD_PUNC, N_NST, RP_NEG, DM_DMD, N_NN...","[V_VM_VF, RD_PUNC, N_NST, RP_NEG, PR_PRP, N_NN..."
1,दोन तीन दिसांनी जायत सुरू येरादारी . इतली कित्...,दोन\QT_QTC तीन\QT_QTC दिसांनी\N_NN जायत\V_VM_V...,दोन\QT_QTC तीन\QT_QTC दिसांनी\N_NN जायत\V_VM_V...,11,11,0.818182,0.909091,0.818182,0.854545,"[QT_QTC, QT_QTC, N_NN, V_VM_VNF, RB, N_NN, RD_...","[QT_QTC, QT_QTC, N_NN, V_VM_VNF, RB, N_NN, RD_..."
2,आनी जाली ना जाल्यार ? कोण घेतलो आमकां घरांत ? ...,आनी\CC_CCD जाली\V_VM_VNF ना\RP_NEG जाल्यार\CC_...,आनी\CC_CCD जाली\V_VM_VF ना\RP_NEG जाल्यार\CC_C...,22,22,0.863636,0.924242,0.863636,0.863636,"[CC_CCD, V_VM_VNF, RP_NEG, CC_CCS, RD_PUNC, PR...","[CC_CCD, V_VM_VF, RP_NEG, CC_CCS, RD_PUNC, PR_..."
3,"हांव अशें आसा म्हणून तर सांगतां , घाई नाका . आ...",हांव\PR_PRP अशें\DM_DMD आसा\V_VM_VF म्हणून\CC_...,हांव\PR_PRP अशें\DM_DMD आसा\V_VM_VF म्हणून\V_V...,17,17,0.882353,1.000000,0.882353,0.921569,"[PR_PRP, DM_DMD, V_VM_VF, CC_CCS, CC_CCS, V_VM...","[PR_PRP, DM_DMD, V_VM_VF, V_VM_VNF, CC_CCS, V_..."
4,पूण हांगा रावनय जावचें ना . चलत रावल्यार मेळत ...,पूण\CC_CCS हांगा\N_NST रावनय\V_VM_VNF जावचें\V...,पूण\CC_CCS हांगा\N_NST रावनय\N_NN जावचें\V_VM_...,15,15,0.933333,0.966667,0.933333,0.941414,"[CC_CCS, N_NST, V_VM_VNF, V_VM_VNF, RP_NEG, RD...","[CC_CCS, N_NST, N_NN, V_VM_VNF, RP_NEG, RD_PUN..."


In [67]:
df_anwani.to_excel("test_svm_100_anwani.xlsx", index=False)

In [68]:
# Flatten all reference and predicted labels for classification report
all_ref_labels = [label for labels in df_anwani["Ref Labels"] for label in labels]
all_pred_labels = [label for labels in df_anwani["Pred Labels"] for label in labels]

# Generate classification report
print("\nClassification Report:\n")
print(classification_report(all_ref_labels, all_pred_labels, zero_division=0))


Classification Report:

              precision    recall  f1-score   support

      CC_CCD       1.00      1.00      1.00        13
      CC_CCS       0.61      0.79      0.69        43
      DM_DMD       0.92      0.85      0.88        52
      DM_DMI       1.00      0.70      0.82        10
      DM_DMQ       1.00      0.07      0.12        15
      DM_DMR       0.00      0.00      0.00         0
          JJ       0.30      0.67      0.41        12
         NST       0.00      0.00      0.00         1
        N_NN       0.78      0.95      0.86       201
       N_NNP       0.00      0.00      0.00         1
       N_NST       0.97      0.85      0.90        78
      PR_PRF       1.00      1.00      1.00         1
      PR_PRI       1.00      0.36      0.53        11
      PR_PRP       0.95      0.95      0.95       131
      PR_PRQ       0.54      0.75      0.63        20
         PSP       0.79      0.58      0.67        38
      QT_QTC       0.88      0.94      0.91        16
  